
# Pipeline completo — Preparação e redução de dimensionalidade

Este notebook executa **as mesmas 13 etapas** do pipeline de código (`run_pipeline.py`),
chamando as mesmas funções de `src/`. Qualquer um dos dois pode ser executado
independentemente e produz os mesmos artefatos (figuras, tabelas, relatórios, modelo).

**Sequência:** load → prep → eda → testes → seleção → PCA → modelagem → redução →
modelos supervisionados → relatório de etapas → relatório → relatório de modelos → dicionário.

Cada etapa responde à pergunta orientadora: *que problema nos dados justifica esta
transformação?* (cf. atividade da disciplina).


## 0. Ambiente e imports

In [1]:

from pathlib import Path
import pandas as pd
import json
import sys

raiz = Path.cwd()
while raiz != raiz.parent and not (raiz / "src" / "config.py").exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

from src.config import (
    ANALYSIS_CSV, STATUS_CSV, FIGURES, TABLES, OUTPUTS,
    ETAPAS_MD, REPORT_MD, DICTIONARY_MD,
)

print("Raiz do repositório:", raiz.resolve())
print("Diretório de trabalho:", Path.cwd().resolve())
print("Saídas em:", OUTPUTS.resolve())
print("Figuras em:", FIGURES.resolve())
print("Tabelas em:", TABLES.resolve())


Raiz do repositório: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project
Diretório de trabalho: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\notebooks
Saídas em: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\outputs
Figuras em: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\reports\figures
Tabelas em: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\reports\tables



## 1. Etapa load — construção da base analítica

**O que faz:** baixa/ler as 9 tabelas do dataset Olist, junta por `order_id`, cria
atributos derivados (`atraso_dias`, `entrega_dias`, `frete_ratio`, `distancia_km`,
`n_itens`, `mes_compra`, …) e faz o recorte de pedidos **entregues com avaliação**.

**Por que:** só pedidos entregues e avaliados têm alvo (`review_score`) e permitem
calcular atraso real vs. prometido. Resultado: **95.824 pedidos** na base de análise.


In [2]:

from src.features import salvar_bases

salvar_bases()

print("ANALYSIS_CSV existe:", ANALYSIS_CSV.exists(), "| linhas:", len(pd.read_csv(ANALYSIS_CSV)))
print("STATUS_CSV existe:", STATUS_CSV.exists())


[data] olist_orders_dataset.csv: 99,441 linhas x 8 colunas
[data] olist_customers_dataset.csv: 99,441 linhas x 5 colunas


[data] olist_order_items_dataset.csv: 112,650 linhas x 7 colunas


[data] olist_order_reviews_dataset.csv: 99,224 linhas x 7 colunas
[data] olist_order_payments_dataset.csv: 103,886 linhas x 5 colunas


[data] olist_products_dataset.csv: 32,951 linhas x 9 colunas
[data] olist_sellers_dataset.csv: 3,095 linhas x 4 colunas


[data] olist_geolocation_dataset.csv: 1,000,163 linhas x 5 colunas
[data] product_category_name_translation.csv: 71 linhas x 2 colunas


[features] base de análise: 95,824 pedidos entregues com review | insatisfeitos (nota<=2): 12.8%


[features] salvo -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\outputs\analysis_dataset.csv
[features] salvo -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\outputs\orders_status_dataset.csv


ANALYSIS_CSV existe: True | linhas: 95824
STATUS_CSV existe: True



## 2. Etapa prep — pré-processamento e diagnóstico de qualidade

**O que faz:**
1. Detecta valores impossíveis/incoerentes (regras de domínio) e os torna ausentes.
2. Diagnostica duplicatas, constantes, identificadores, redundâncias e colinearidade.
3. Remove variáveis derivadas exatas (`valor_pedido`, `preco_unitario_medio`).
4. Divide estratificado 80/20 (seed 42) e imputa ausentes com mediana/moda **apenas do treino**.
5. Codifica categóricas (one-hot) e escalona numéricas; salva matrizes em `outputs/prep/`.

**Por que:** imputar/escalonar só no treino evita vazamento de estatísticas do teste;
dividir antes de qualquer transformação protege a avaliação final.


In [3]:

from src.qualidade import executar_etapa_prep

executar_etapa_prep()

import json
prep = json.loads((OUTPUTS / "prep" / "prep_info.json").read_text(encoding="utf-8"))
print("Treino:", prep["n_train"], "| Teste:", prep["n_test"])
print("Dimensão saída:", prep["n_dim_saida"], "colunas")
print("Inválidos tratados:", prep["n_invalidos_total"])


[prep] base: 95,824 pedidos | 42 colunas


[prep] tabela -> tab15_diagnostico_qualidade.csv (51 verificações)
[prep]   inválido: manuseio_dias: postagem anterior à compra -> 134
[prep]   inválido: manuseio_dias: postagem posterior à entrega ao cliente -> 23
[prep]   inválido: n_parcelas: pagamento com 0 parcelas (mínimo possível = 1) -> 2
[prep]   inválido: peso_g: peso do produto ≤ 0 g -> 6
[prep]   inválido: distancia_km: distância > 4500 km (fora do território brasileiro) -> 4


[prep] figura -> fig11_valores_impossiveis.png
[prep] VIF máx.: 1000000.0 (antes) -> 2.7 (depois da remoção das redundantes)


[prep] base final: treino (76659, 142) / teste (19165, 142) -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\outputs\prep
[prep] etapa concluída.
Treino: 76659 | Teste: 19165
Dimensão saída: 142 colunas
Inválidos tratados: 169



## 3. Etapa EDA — análise exploratória dos dados

**O que faz:** gera as figuras de análise na sequência lógica do projeto:
distribuição das notas, evolução temporal, atraso por nota, insatisfação por faixa de
atraso, categorias críticas, insatisfação por UF e o **scatterplot** distância × frete.

**Por que:** entender o alvo (notas 1-2 = 12,8% insatisfeitos) e quais fatores
operacionais se associam à insatisfação antes de modelar.


In [4]:

from src.eda import executar_etapa_eda

executar_etapa_eda()

from pathlib import Path
figs = sorted(Path("reports/figures").glob("*.png"))
print(f"{len(figs)} figuras geradas até agora:")
for f in figs:
    print("  -", f.name)


[eda] analisando 95,824 pedidos...


[eda] figura -> fig01_distribuicao_notas.png


[eda] figura -> fig02_evolucao_temporal.png


[eda] figura -> fig03_atraso_por_nota.png


[eda] figura -> fig04_insatisfacao_por_atraso.png


[eda] figura -> fig05_categorias.png


[eda] figura -> fig06_insatisfacao_por_uf.png


[eda] figura -> fig07_dispersao_frete_distancia.png
[eda] tabela -> tab01_resumo_notas.csv
[eda] tabela -> tab02_kpis_gerais.csv
[eda] tabela -> tab03_insatisfacao_por_categoria.csv
[eda] tabela -> tab04_insatisfacao_por_uf.csv
[eda] etapa concluída.
0 figuras geradas até agora:



## 4. Etapa testes — hipóteses estatísticas

**O que faz:** testa associações entre variáveis e a insatisfação usando Mann-Whitney
(numéricas), qui-quadrado (categóricas) e Spearman (correlações), reportando p-valor
**e** tamanho de efeito.

**Por que:** com N ≈ 96 mil, diferenças mínimas tornam-se "significativas"; o tamanho
de efeito diz se a diferença é relevante na prática, não só estatisticamente.


In [5]:

from src.statistics_tests import executar_etapa_testes

executar_etapa_testes()

tab = pd.read_csv(TABLES / "tab05_testes_hipotese.csv")
print(tab[["comparacao", "teste", "p_valor", "medida_efeito", "valor_efeito"]].head(12))


[testes] tabela -> tab05_testes_hipotese.csv (19 testes)
[testes] tabela -> tab06_odds_ratio_atraso.csv
[testes] OR do atraso = 11.593 (IC95% 11.023-12.192)
[testes] etapa concluída.
                                     comparacao  \
0   Atraso (dias) — insatisfeitos x satisfeitos   
1                 Tempo total de entrega (dias)   
2                              Frete total (R$)   
3           Participação do frete no pedido (%)   
4                          Valor do pedido (R$)   
5                         Nº de itens do pedido   
6               Distância cliente-vendedor (km)   
7                   Tempo até a postagem (dias)   
8                      Nota x atraso na entrega   
9                 Nota x tempo total de entrega   
10                 Nota x participação do frete   
11            Nota x distância cliente-vendedor   

                              teste        p_valor  \
0                    Mann-Whitney U   0.000000e+00   
1                    Mann-Whitney U   0.00000


## 5. Etapa seleção — consenso de 5 métodos

**O que faz:** aplica cinco métodos de seleção (dois filtros: informação mútua e AUC
univariada; um wrapper: forward com regressão logística; dois embutidos: L1 e Random
Forest). Cada método compara cada variável com cópias embaralhadas (features-sombra).
Entra quem alcançar ≥ 3 de 5 votos.

**Por que:** reduzir de 19 features para as que carregam sinal real, preservando
interpretabilidade (odds ratio e SHAP por variável) e desempenho preditivo.


In [6]:

from src.feature_selection import executar_etapa_selecao

executar_etapa_selecao()

sel = json.loads((OUTPUTS / "selecao_atributos.json").read_text(encoding="utf-8"))
print("Selecionadas:", len(sel["selecionadas"]), "de", 19)
print("Mantidas:", sel["selecionadas"])


[select] seleção usa SOMENTE o treino: 76,659 pedidos, 19 features


[select] filtro MI: 5s


[select] filtro AUC univariada: 10s


[select]   wrapper passo 1: +entrega_dias (AUC val. 0.6773, ganho +0.1773)


[select]   wrapper passo 2: +n_itens (AUC val. 0.7151, ganho +0.0378)


[select]   wrapper passo 3: +distancia_km (AUC val. 0.7222, ganho +0.0071)


... (5 outputs left out)

[select] tabela -> tab12_ranking_features.csv


[select] figura -> fig08_ranking_spearman.png


[select] figura -> fig09_mutual_information.png


[select] figura -> fig10_consenso_selecao.png
[select] consenso (>= 3/5): 11 de 19 features
[select]   mantidas:   ['entrega_dias', 'n_itens', 'atraso_dias', 'manuseio_dias', 'categoria_grupo', 'distancia_km', 'preco_total', 'mes_compra', 'frete_total', 'uf_cliente', 'peso_g']
[select]   descartadas: ['n_fotos', 'n_parcelas', 'uf_vendedor', 'frete_ratio', 'aprovacao_h', 'pagamento_tipo', 'hora_compra', 'dia_semana_compra']
[select] etapa concluída.
Selecionadas: 11 de 19
Mantidas: ['entrega_dias', 'n_itens', 'atraso_dias', 'manuseio_dias', 'categoria_grupo', 'distancia_km', 'preco_total', 'mes_compra', 'frete_total', 'uf_cliente', 'peso_g']



## 6. Etapa PCA — redução de dimensionalidade

**O que faz:** ajusta PCA **só no treino** sobre as numéricas padronizadas, compara
critérios (Kaiser, análise paralela, variância acumulada) e valida o número de
componentes `k` pela informação preditiva preservada (AUC de validação cruzada).

**Por que:** não se mantém componentes "até atingir X% da variância"; o k final vem do
desempenho preditivo, não de uma regra arbitrária de variância.


In [7]:

from src.pca_diagnostico import executar_pca

executar_pca()

pca = json.loads((OUTPUTS / "pca_info.json").read_text(encoding="utf-8"))
print("k escolhido:", pca["k_escolhido"])
print("AUC sem PCA:", round(pca["auc_cv_sem_pca"], 4),
      "| AUC com k escolhido:", round(pca["auc_cv_k_escolhido"], 4))


[pca] ajustado SOMENTE no treino (76,659 pedidos), 13 numéricas padronizadas


[pca] validando k por desempenho preditivo (LR, CV 3 dobras)...


[pca]   k= 1: AUC CV = 0.6766 ± 0.0035


[pca]   k= 2: AUC CV = 0.7087 ± 0.0041


[pca]   k= 3: AUC CV = 0.7087 ± 0.0040


[pca]   k= 4: AUC CV = 0.7082 ± 0.0044


... (6 outputs left out)

[pca]   k=11: AUC CV = 0.7283 ± 0.0050


[pca]   k=12: AUC CV = 0.7309 ± 0.0051


[pca]   k=13: AUC CV = 0.7341 ± 0.0054


[pca] Kaiser: 5 | análise paralela: 5 | 90% da variância: 10 | validação cruzada (1 EP): 13
[pca] k escolhido = 13 (100.0% da variância; AUC CV 0.7341 vs. 0.7341 sem PCA)
[pca] etapa concluída.
k escolhido: 13
AUC sem PCA: 0.7341 | AUC com k escolhido: 0.7341



## 7. Etapa model — modelos de classificação

**O que faz:** treina regressão logística (baseline interpretável) e gradient boosting
(HGB) com validação cruzada estratificada 5 dobras. Escolhe o limiar de decisão pelo
F1 máximo fora-da-dobra (não 0,5, por classe rara). Avalia com **acurácia, precisão,
recall, F1 e AUC** (sem PR-AUC, Brier ou ECE). Gera curvas ROC, matriz de confusão,
calibração e importância SHAP.

**Por que:** comparar dois modelos no mesmo protocolo e ficar com o que melhor separa
insatisfeitos de satisfeitos; a LR serve como régua interpretável.


In [8]:

from src.model import executar_etapa_modelo

executar_etapa_modelo()

modelo = json.loads((OUTPUTS / "resultados_modelo.json").read_text(encoding="utf-8"))
print("Melhor modelo:", modelo["melhor_modelo"])
for nome, m in modelo["metricas"].items():
    print(f"  {nome:24s} AUC={m['roc_auc']}  acurácia={m['acuracia']}  "
          f"precisão={m['precisao']}  recall={m['recall']}  F1={m['f1']}  limiar={m['limiar']}")


[model] 95,824 amostras | positivos (nota 1-2): 12.8% | 19 features brutas | treino 76,659 / teste 19,165
[model] validação cruzada (5x) - regressao_logistica...


[model] regressao_logistica: AUC test=0.7404 | limiar(F1, OOF)=0.2425 | acurácia=0.8687 | precisão=0.4834 | recall=0.3862 | F1=0.4294
[model] validação cruzada (5x) - gradient_boosting...


[model] gradient_boosting: AUC test=0.7649 | limiar(F1, OOF)=0.2294 | acurácia=0.8724 | precisão=0.5016 | recall=0.4531 | F1=0.4761
[model] melhor modelo por AUC de validação cruzada (não pelo teste): gradient_boosting


[model] figura -> fig16_curvas_roc.png
[model] figura -> fig17_matriz_confusao.png


[model] figura -> fig18_calibracao.png
[model] tabela -> tab07_odds_ratios_logistica.csv (142 variáveis)


[model] tabelas -> tab08_importancia_permutacao.csv, tab20_importancia_grupos.csv
[model] tabela -> tab09_metricas_modelos.csv


[model] figura -> fig15_shap_importancia.png


[model] validação temporal (gradient_boosting): AUC=0.6942


[model] modelo pré-entrega (gradient_boosting): AUC=0.692
[model] modelo salvo -> modelo_insatisfacao.joblib
[model] resultados -> resultados_modelo.json
[model] etapa concluída.
Melhor modelo: gradient_boosting
  regressao_logistica      AUC=0.7404  acurácia=0.8687  precisão=0.4834  recall=0.3862  F1=0.4294  limiar=0.2425
  gradient_boosting        AUC=0.7649  acurácia=0.8724  precisão=0.5016  recall=0.4531  F1=0.4761  limiar=0.2294



## 8. Etapa redução — comparação de representações

**O que faz:** treina LR e HGB nas três representações (original, selecionada, PCA)
com os mesmos dados, divisão e validação, e compara AUC, F1, número de colunas e
interpretabilidade.

**Por que:** verificar se a redução (seleção ou PCA) preserva a informação preditiva.
Resultado esperado: seleção mantém o AUC (Δ ≈ −0,0001) com menos colunas; o PCA
prejudica o desempenho (Δ −0,009 a −0,020).


In [9]:

from src.reducao import executar_etapa_reducao

executar_etapa_reducao()

tab = pd.read_csv(TABLES / "tab16_comparacao_representacoes.csv")
print(tab[["representacao", "modelo", "n_colunas_modelo", "auc_teste", "f1_teste"]])


[reducao] Original     | regressao_logistica  | 142 colunas | AUC CV 0.7341 | AUC teste 0.7404 | F1 0.427 | acc 0.861 | 2.8s


[reducao] Original     | gradient_boosting    | 142 colunas | AUC CV 0.7599 | AUC teste 0.7649 | F1 0.478 | acc 0.876 | 2.2s


[reducao] Selecionada  | regressao_logistica  | 109 colunas | AUC CV 0.7336 | AUC teste 0.7386 | F1 0.428 | acc 0.864 | 2.3s


[reducao] Selecionada  | gradient_boosting    | 109 colunas | AUC CV 0.7582 | AUC teste 0.7669 | F1 0.478 | acc 0.874 | 1.6s


[reducao] PCA          | regressao_logistica  | 142 colunas | AUC CV 0.7341 | AUC teste 0.7404 | F1 0.427 | acc 0.860 | 2.7s


[reducao] PCA          | gradient_boosting    | 142 colunas | AUC CV 0.7487 | AUC teste 0.7581 | F1 0.452 | acc 0.864 | 2.8s
[reducao] tabela -> tab16_comparacao_representacoes.csv


[reducao] figura -> fig19_comparacao_representacoes.png


[reducao] figura -> fig20_roc_representacoes.png
[reducao] etapa concluída.
  representacao               modelo  n_colunas_modelo  auc_teste  f1_teste
0      Original  regressao_logistica               142   0.740420  0.426971
1      Original    gradient_boosting               142   0.764915  0.477632
2   Selecionada  regressao_logistica               109   0.738552  0.427946
3   Selecionada    gradient_boosting               109   0.766931  0.478477
4           PCA  regressao_logistica               142   0.740420  0.427471
5           PCA    gradient_boosting               142   0.758125  0.452035



## 8b. Etapa modelos — modelos supervisionados e avaliação

**O que faz:** compara cinco famílias de modelos (regressão logística, árvore de decisão, KNN,
gradient boosting e MLP) com busca de hiperparâmetros por validação cruzada estratificada de 5
dobras **somente no treino**, com as mesmas dobras para todos. O limiar de decisão minimiza o
custo esperado com **falso negativo = 3 × falso positivo** (o cliente insatisfeito pesa mais).
O teste é usado uma única vez; os IC95% vêm de bootstrap.

**Por que:** responder *qual modelo é o mais adequado e que evidências sustentam a escolha*,
olhando matriz de confusão, precisão, recall, F1 e AUC, e não só a acurácia (o classificador que
sempre prevê "satisfeito" já acerta 87,2%).

**Tempo:** cerca de 15 minutos. Para um teste rápido com grades mínimas, defina
`MODELOS_RAPIDO=1` antes de abrir o Jupyter.


In [10]:

from src.modelos_supervisionados import executar_etapa_modelos

executar_etapa_modelos()

mod = json.loads((OUTPUTS / "resultados_modelos_supervisionados.json").read_text(encoding="utf-8"))
tab = pd.read_csv(TABLES / "tab24_comparacao_modelos.csv")
print("Modelo final:", mod["nome_final"], "|", mod["criterio_final"])
print(tab[["modelo", "limiar", "acuracia", "precisao", "recall", "f1", "auc_teste", "auc_cv"]].round(3).to_string(index=False))


[modelos] treino 76,659 / teste 19,165 | positivos 12.8% | grades completas | custo FN:FP = 3:1
[modelos] Regressão logística: 4 configurações x 5 dobras...


[modelos]   melhor: {'C': '0.1'} | AUC CV 0.7344 | busca 23s


[modelos] Árvore de decisão: 56 configurações x 5 dobras...


[modelos]   melhor: {'criterion': 'gini', 'max_depth': '10', 'min_samples_leaf': '200'} | AUC CV 0.7433 | busca 137s


[modelos] KNN: 20 configurações x 5 dobras...


[modelos]   melhor: {'n_neighbors': '301', 'weights': 'distance', 'pre': 'selecionadas (11 variáveis)'} | AUC CV 0.7475 | busca 302s


... (8 outputs left out)

[modelos] figura -> fig24_curvas_validacao.png


[modelos] figura -> fig25_overfitting_auc.png


[modelos] figura -> fig26_arvore_regras.png


[modelos] figura -> fig27_sensibilidade_custo.png
[modelos] resultados -> resultados_modelos_supervisionados.json | modelo final salvo -> modelo_supervisionado_final.joblib
[modelos] etapa concluída em 953s.
Modelo final: Gradient boosting | entre os modelos com AUC de CV a até 1 erro-padrão do melhor (Gradient boosting: 0.7610 - 0.0028), menor custo esperado em CV com FN = 3 x FP
             modelo  limiar  acuracia  precisao  recall    f1  auc_teste  auc_cv
regressao_logistica   0.255     0.874     0.508   0.374 0.431      0.741   0.734
     arvore_decisao   0.265     0.875     0.513   0.408 0.454      0.748   0.743
                knn   0.218     0.866     0.476   0.442 0.458      0.756   0.747
  gradient_boosting   0.247     0.879     0.536   0.430 0.477      0.765   0.761
                mlp   0.240     0.871     0.493   0.438 0.464      0.759   0.752



## 9. Etapa etapas — relatório detalhado por etapa

**O que faz:** gera `reports/relatorio_etapas.md`, documentando cada decisão com a
respectiva evidência e justificativa de domínio.


In [11]:

from src.etapas import gerar_relatorio_etapas

gerar_relatorio_etapas()
print("Relatório de etapas:", ETAPAS_MD)


[etapas] relatório -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\relatorio_etapas.md
Relatório de etapas: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\relatorio_etapas.md



## 10. Etapa report — relatório analítico

**O que faz:** gera `reports/relatorio.md` com toda a narrativa do trabalho, as 20
figuras organizadas por fase (análise → seleção → prep → PCA → modelagem → avaliação)
e as métricas permitidas.


In [12]:

from src.report import gerar_relatorio

gerar_relatorio()
print("Relatório:", REPORT_MD)


[report] relatório -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\relatorio.md
Relatório: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\relatorio.md



## 10b. Etapa relatorio_modelos — relatório de modelos supervisionados

**O que faz:** gera `reports/relatorio_modelos.md` na estrutura pedida pela atividade: problema e
variável-alvo, amostragem e validação, modelos e hiperparâmetros, avaliação, escolha do modelo
final e limitações. O texto é montado a partir dos números da etapa `modelos`.


In [13]:

from src.config import RELATORIO_MODELOS_MD
from src.relatorio_modelos import gerar_relatorio_modelos

gerar_relatorio_modelos()
print("Relatório de modelos:", RELATORIO_MODELOS_MD)


[relatorio_modelos] relatório -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\relatorio_modelos.md
Relatório de modelos: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\relatorio_modelos.md



## 11. Etapa dicionário — dicionário de dados

**O que faz:** gera `reports/dicionario_dados.md` documentando as tabelas brutas do
dataset, as colunas da base final e seu papel (feature / alvo / descartada).


In [14]:

from src.report import gerar_dicionario

gerar_dicionario()
print("Dicionário:", DICTIONARY_MD)


[data] olist_orders_dataset.csv: 99,441 linhas x 8 colunas
[data] olist_customers_dataset.csv: 99,441 linhas x 5 colunas


[data] olist_order_items_dataset.csv: 112,650 linhas x 7 colunas


[data] olist_order_reviews_dataset.csv: 99,224 linhas x 7 colunas
[data] olist_order_payments_dataset.csv: 103,886 linhas x 5 colunas
[data] olist_products_dataset.csv: 32,951 linhas x 9 colunas
[data] olist_sellers_dataset.csv: 3,095 linhas x 4 colunas


[data] olist_geolocation_dataset.csv: 1,000,163 linhas x 5 colunas
[data] product_category_name_translation.csv: 71 linhas x 2 colunas


[report] dicionário -> C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\dicionario_dados.md
Dicionário: C:\Users\yagom\Documents\GitHub\Data-Science-Class-Project\docs\dicionario_dados.md



## Conclusão

O notebook executou as **13 mesmas etapas** do pipeline de código. Como ambos chamam as
mesmas funções de `src/`, são **versões intercambiáveis**: executar este notebook ou
`python3 run_pipeline.py` gera os mesmos artefatos.

Resultado central: o **atraso na entrega** é o fator dominante da insatisfação
(odds ratio ≈ 11,6×). A seleção de atributos (11 de 19 features) preserva o desempenho,
enquanto o PCA o reduz. O modelo final (HGB) atinge AUC ≈ 0,766 e F1 ≈ 0,478 no teste.

Na comparação de modelos supervisionados (regressão logística, árvore, KNN, gradient boosting e MLP), o gradient boosting segue como modelo final (AUC 0,766 no teste; recall 0,425 e precisão 0,540 com falso negativo = 3 × falso positivo). As cinco famílias ficam numa faixa estreita de AUC (0,741–0,766): o teto parece vir da informação das variáveis, não do algoritmo.
